# 04 · Gaps against should-cost and sensitivity

Every purchase order line and quote compared with its should-cost at the same date (copper lagged 3 months, labour priced in the supplier's country). A supplier is flagged when its spend-weighted gap is **material** (above 10%) and an **outlier** (more than two robust standard deviations, MAD, above the median supplier).

The rules live in `src/gap.py`; the should-cost premises in `src/should_cost.py`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd

from src import gap
from src import should_cost as sc
from src.db import DEFAULT_DB, connect, create_schema

con = connect(DEFAULT_DB)
create_schema(con)

## Supplier gaps

In [ ]:
suppliers = gap.supplier_gaps(con)
print(f"Median supplier gap: {suppliers['gap'].median():.1%}")
suppliers.head(10).assign(
    spend_usd_m=lambda d: (d["spend_usd"] / 1e6).round(2),
    gap=lambda d: d["gap"].round(3),
    z=lambda d: d["z"].round(2),
)[["supplier_id", "country", "materials", "spend_usd_m", "gap", "z", "flagged"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(suppliers["gap"], bins=25, color="tab:grey")
for s in suppliers[suppliers["flagged"]].itertuples():
    ax.axvline(s.gap, color="tab:red", lw=1)
    ax.text(s.gap, ax.get_ylim()[1] * 0.9, f" {s.supplier_id}", color="tab:red", fontsize=8)
ax.axvline(gap.MATERIALITY, ls="--", color="black", lw=0.8)
ax.set(
    xlabel="Spend-weighted gap to should-cost",
    ylabel="Suppliers",
    title="Distribution of supplier gaps (red: flagged)",
);

## Quotes

The latest RFQs, three quotes each, checked against the same should-cost. A quote ranking that matches the PO ranking means the gap is in the supplier's price level, not in one-off orders.

In [ ]:
quotes = gap.quote_gaps(con)
quotes.head(8).assign(gap=lambda d: d["gap"].round(3))

## Savings potential

Spend in the last 12 months above should-cost, for material-supplier pairs whose gap is above 10%.

In [ ]:
since = date(2025, 11, 1)
pairs = gap.savings_potential(con, since)
flagged_ids = set(suppliers.loc[suppliers["flagged"], "supplier_id"])
total = pairs["savings_usd"].sum()
on_flagged = pairs.loc[pairs["supplier_id"].isin(flagged_ids), "savings_usd"].sum()
print(
    f"{len(pairs)} material-supplier pairs above 10%; savings potential USD {total:,.0f} "
    f"per year, {on_flagged / total:.0%} of it with the flagged suppliers"
)
pairs.groupby("supplier_id")["savings_usd"].sum().sort_values(ascending=False).head(8).round(0)

## Sensitivity

One driver at a time for a body harness made in Brazil and in China: cost drivers ±20%, percentage rates ±3 points.

In [ ]:
copper = con.execute(
    "SELECT avg(copper_usd_t) FROM (SELECT copper_usd_t FROM copper_monthly "
    "ORDER BY month DESC LIMIT 12)"
).fetchone()[0]
tables = {}
for country in ("BR", "CN"):
    t = gap.tornado(sc.BY_FAMILY["harness_body"], copper, country)
    tables[country] = t.assign(low_pct=t["low"] / t["base"] - 1, high_pct=t["high"] / t["base"] - 1)
pd.concat(tables, names=["made_in"])[["driver", "base", "low_pct", "high_pct"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, (country, t) in zip(axes, tables.items(), strict=True):
    t = t.iloc[::-1]
    ax.barh(t["driver"], t["low_pct"], color="tab:green")
    ax.barh(t["driver"], t["high_pct"], color="tab:red")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"Body harness made in {country} (base USD {t['base'].iloc[0]:.0f})")
    ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()

## Reading

- Regular suppliers price a median 4.7% above should-cost, a reasonable error band for premises built from public data.
- **S019, S032 and S023** are flagged: gaps of 21.6%, 18.5% and 11.7%, all well outside the spread of the other suppliers, and their quotes rank the same way. They are the negotiation targets.
- **S042** is above 10% but not an outlier (z just under 2). It is a watch item: ask for a cost breakdown at the next RFQ rather than open a renegotiation.
- The savings potential is about **USD 0.83 M per year**, 72% of it with the three flagged suppliers.
- In Brazil, **other material and copper** move the should-cost the most, then labour. In China, labour weighs less and copper and material weigh more. A copper index clause therefore protects most of the price risk; margin is a small lever.

Next: which copper clause to put in the contract (backtest of four designs).